# Figures for Maaike Data
Dataset can be downloaded from https://www.ebi.ac.uk/pride/archive/projects/PXD010990 and paper from https://doi.org/10.1016/j.ccell.2018.09.009.

## Preparation

In [ ]:
import os,csv,random
import pandas as pd
import numpy as np
import scanpy as sc
import math

from skimage import io, color
import torch

In [ ]:
from scanpy import read_10x_h5
import SpaGCN as spg
import matplotlib.pyplot as plt
import json
from tqdm import tqdm
import pickle

In [ ]:
from pyimzml.ImzMLParser import ImzMLParser
from pyimzml.metadata import Metadata

In [ ]:
# === CONFIGURE ME ===
DATA_DIR = 'data'        # input data root (subdirectories per dataset live underneath)
OUTPUT_DIR = 'output_Maaike'    # where this notebook writes its outputs
REPO_ROOT = '.'                # any remaining absolute-path references resolve to here
# ====================


In [ ]:
import GalaxyPython as gx
gx.__version__

## Reading Data
import Ds1, (2), 3, 4, 18, 19, 20, 24, (26) (normal)

In [ ]:
weekpoint1 = "5 wk regression"
weekpoint2 = "2 wk regression"

# compare within pos/neg group
# group = "9AA neg" 
group = "DHB pos"

DataDir = "data/Maaike"

In [ ]:
MALDIdata1 = pd.read_csv("{DataDir}/{weekpoint} - {group} - All Spectra.csv".format(weekpoint = weekpoint1, DataDir = DataDir, group = group), sep =';') 
MALDIloc1 = pd.read_csv("{DataDir}/{weekpoint} - {group} - Region Spots.csv".format(weekpoint = weekpoint1, DataDir = DataDir, group = group), sep =';') 
MALDIdata2 = pd.read_csv("{DataDir}/{weekpoint} - {group} - All Spectra.csv".format(weekpoint = weekpoint2, DataDir = DataDir, group = group), sep =';') 
MALDIloc2 = pd.read_csv("{DataDir}/{weekpoint} - {group} - Region Spots.csv".format(weekpoint = weekpoint2, DataDir = DataDir, group = group), sep =';') 

In [ ]:
mz_wk5 = pd.DataFrame(list(MALDIdata1.columns[1:]), index = list(MALDIdata1.columns[1:])).astype('float')
mz_wk5 = mz_wk5.rename(columns = {0:"m/z"})

mz_wk2 = pd.DataFrame(list(MALDIdata2.columns[1:]), index = list(MALDIdata2.columns[1:])).astype('float')
mz_wk2 = mz_wk2.rename(columns = {0:"m/z"})

In [ ]:
# pandas.dataset.iloc(row, column) is used for retrive rows and columns from a dataset
MALDIdataAnn1 = sc.AnnData(X = MALDIdata1.iloc[:,1:], var = mz_wk5, obs = MALDIloc1)
MALDIdataAnn2 = sc.AnnData(X = MALDIdata2.iloc[:,1:], var = mz_wk2, obs = MALDIloc2)

In [ ]:
sc.pp.normalize_per_cell(MALDIdataAnn1)
sc.pp.normalize_per_cell(MALDIdataAnn2)

## Plots of m/z shift

### Using code from MALDI package

In [ ]:
PeakGroup = gx.PeakCallingmv(MALDIdataAnn1, MALDIdataAnn2)
PeakGroup.peak_calling(0.9)
PeakGroup.peak_grouping(0.9)

In [ ]:
ExactAlign = gx.AnnDataMALDI(MALDIdataAnn1, MALDIdataAnn2)
ExactAlign.get_corr_peakgroup_refined(PeakGroup.jointcluster)
ExactAlign.peak_group_pairing()
ExactAlign.fine_alignment_assessment(threshold= 0.2, ignore = True)
ExactAlign.summarize()

In [ ]:
## Shifting distribution plot
SD = gx.MALDI_SIM(ExactAlign)
file_loc = f'{REPO_ROOT}/'
SD.shiftdatadf.to_csv("output_Maaike/mzshifting.csv", sep = ",")

### plot of m/z shift in m/z unit

In [ ]:
Aligned = gx.PGmzalign(ExactAlign)
aligned_data = Aligned.getAnnSim()

In [ ]:
df = Aligned.shiftdatadf

In [ ]:
Aligned.shiftplot_data()

In [ ]:
import anndata as ad
aligned_df = pd.DataFrame(df.X)

In [ ]:
SD.shiftdatadf.to_csv("output_Maaike/aligned.csv", sep = ",")

## Plots of Pearson's Coefficient

In [ ]:
import seaborn as sns

In [ ]:
PeakGroup = gx.PeakCallingmv(MALDIdataAnn1, MALDIdataAnn2)
PeakGroup.peak_calling(0.9)
PeakGroup.peak_grouping(0.9)

In [ ]:
ExactAlign = gx.AnnDataMALDI(MALDIdataAnn1, MALDIdataAnn2)
ExactAlign.get_corr_peakgroup_refined(PeakGroup.jointcluster)
ExactAlign.peak_group_pairing()
ExactAlign.fine_alignment_assessment(threshold= 0.2, ignore = True)
ExactAlign.summarize()

In [ ]:
# PeakGroup.jointcluster

In [ ]:
# ExactAlign.PearsonMatrixFull

In [ ]:
num_rows, num_columns = ExactAlign.PearsonMatrixFull.shape
num_rows
num_columns

In [ ]:
ExactAlign.nclusters

In [ ]:
1782/198

In [ ]:
sns.heatmap(ExactAlign.PearsonMatrixFull[300:400, 300:400], cmap = custom_cmap, square = True)
plt.show()

## Anchor point 1

In [ ]:
len(ExactAlign.aligned_mz_clusters_unk)
len(ExactAlign.aligned_mz_clusters_ref)
ExactAlign.nclusters

In [ ]:
# df = ExactAlign.mz_valueUnk.iloc[ExactAlign.aligned_mz_clusters_unk[43]]
43*9

In [ ]:
# ExactAlign.aligned_mz_clusters_ref[43]

In [ ]:
PeakGroup.mz_valueUnk_mv.iloc[1464]
# MALDIdataAnn1.var.iloc[1465]

In [ ]:
PeakGroup.peakUnk[1464]

In [ ]:
# PeakGroup.clusterUnk[31]

In [ ]:
PeakGroup.mz_valueRef_mv.iloc[1489]
# MALDIdataAnn1.var.iloc[1465]

In [ ]:
PeakGroup.peakRef[1489]

In [ ]:
# PeakGroup.clusterRef[27]

In [ ]:
len(ExactAlign.meanspectrumUnk)

In [ ]:
ExactAlign.unk_clusters[43][52]

In [ ]:
len(ExactAlign.ref_clusters[43])

In [ ]:
ExactAlign.ref_clusters[43][52]

In [ ]:
len(PeakGroup.clusterRef)

In [ ]:
ExactAlign.unk_clusters[43][48:57]

In [ ]:
len(ExactAlign.unk_clusters[43])

In [ ]:
ExactAlign.unk_clusters[43]

In [ ]:
mz_wk2[1438:1556]

In [ ]:
ExactAlign.ref_clusters[42]

In [ ]:
ExactAlign.ref_clusters[43][48:57]

In [ ]:
ExactAlign.ref_clusters[44]

In [ ]:
ExactAlign.unk_clusters[43]

In [ ]:
43*9

In [ ]:
# ExactAlign.mz_valueRef.iloc[ExactAlign.aligned_mz_clusters_ref[43]].iloc[48:57]
# mz_wk2.iloc[ExactAlign.aligned_mz_clusters_ref[43]].iloc[48:57]

In [ ]:
# ExactAlign.mz_valueRef.iloc[ExactAlign.aligned_mz_clusters_ref[43]].iloc[48:57]
# mz_wk2.iloc[ExactAlign.aligned_mz_clusters_ref[43]].iloc[48:57]

In [ ]:
#mz_wk5.iloc[ExactAlign.aligned_mz_clusters_unk[43]].iloc[48:57]

In [ ]:
#ExactAlign.align_group

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Define RGB values (0 to 255) for the red color
red_rgb = (150, 5, 5)

# Convert RGB values to the range expected by Matplotlib (0 to 1)
red_color = tuple(component / 255.0 for component in red_rgb)

# Define a custom colormap
colors = [red_color, 
          (1, 1, 1), 
          red_color]  # Red to white to red
custom_cmap = LinearSegmentedColormap.from_list('custom_red_white_red', colors, N=256)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

y_labels = ['U-4', 'U-3', 'U-2', 'U-1', 'U0', 'U1', 'U2', 'U3', 'U4'] # reference or unknown?
x_labels = ['R-4', 'R-3', 'R-2', 'R-1', 'R0', 'R1', 'R2', 'R3', 'R4']

ax.set_xticklabels(('R-4', '\n194.45', 'R-3', '\n194.57', 'R-2', '\n194.70', 
                     'R-1', '\n194.82', 'R0', '\n194.95', 'R1', '\n195.07', 
                     'R2', '\n195.19', 'R3', '\n195.32', 'R4', '\n195.44'), 
                    ha='center')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

y_labels = ['U-4', 'U-3', 'U-2', 'U-1', 'U0', 'U1', 'U2', 'U3', 'U4'] # reference or unknown?
x_labels = ['R-4', 'R-3', 'R-2', 'R-1', 'R0', 'R1', 'R2', 'R3', 'R4']


#sns.heatmap(ExactAlign.PearsonMatrixFull[369:414, 369:414], cmap = 'RdBu', square = True,
#            xticklabels = x_labels, yticklabels = y_labels, vmin = -1, vmax = 1)# 387:396, 387:396

sns.heatmap(ExactAlign.PearsonMatrixFull[387:396, 387:396], cmap = 'RdBu_r', square = True,
            xticklabels = x_labels, yticklabels = y_labels, vmin = -1, vmax = 1)

# plt.xlabel('Reference', fontsize = 14)
# plt.ylabel('Unknown', fontsize = 14)
# 850 - 900, top left

# Create a y = -x line
x_values = np.arange(len(x_labels) + 1)
y_values = x_values
sns.lineplot(x = x_values, y = y_values, color = 'black', linestyle = '-', linewidth = 1)


plt.suptitle("Anchor Point 1", fontsize = 18, y = 0.95, x = 0.455)
plt.savefig('heatmap_anchor1.png', dpi=300)  # Adjust the filename and dpi as needed
plt.show()

In [ ]:
ExactAlign.mz_valueUnk.iloc[ExactAlign.aligned_mz_clusters_unk[43]].iloc[52, ]

In [ ]:
ExactAlign.mz_valueRef.iloc[ExactAlign.aligned_mz_clusters_ref[43]].iloc[52, ]

In [ ]:
ExactAlign.PearsonMatrix

In [ ]:
ExactAlign.PearsonMatrix[43][43]

In [ ]:
ExactAlign.PearsonMatrixFull[43][43]

In [ ]:
ExactAlign.PearsonMatrixFull[387:396, 387:396]

In [ ]:
diagnallist = [np.nanmean(np.diag(ExactAlign.PearsonMatrixFull[387:396, 387:396], k=i)) for i in range(-4,5)]
diagnallist = [-1 if value!=value else value for value in diagnallist]
diagnallist

In [ ]:
max(diagnallist)

## Anchor point2

In [ ]:
# ExactAlign.mz_valueUnk.iloc[ExactAlign.aligned_mz_clusters_unk[60]]

In [ ]:
# PeakGroup.jointcluster[46]
# 46*9

In [ ]:
# ExactAlign.mz_valueRef.iloc[ExactAlign.aligned_mz_clusters_ref[60]][0:7]
mz_wk2.iloc[ExactAlign.aligned_mz_clusters_ref[60]][0:7]

In [ ]:
# ExactAlign.mz_valueUnk.iloc[ExactAlign.aligned_mz_clusters_unk[60]][0:7]
mz_wk5.iloc[ExactAlign.aligned_mz_clusters_unk[60]][0:7]

In [ ]:
# ExactAlign.unk_clusters[60]
# ExactAlign.ref_clusters[60]

In [ ]:
PeakGroup.clusterRef[41]

In [ ]:
PeakGroup.clusterUnk[54]

In [ ]:
mz_wk2[2115:2500]

In [ ]:
mz_wk5[2077:2500]

In [ ]:
ExactAlign.ref_clusters[60]

In [ ]:
ExactAlign.unk_clusters[60]

In [ ]:
ExactAlign.PearsonMatrixFull[540:549, 540:549]

In [ ]:
y_labels = ['U-4', 'U-3', 'U-2', 'U-1', 'U0', 'U1', 'U2', 'U3', 'U4']
x_labels = ['R-4', 'R-3', 'R-2', 'R-1', 'R0', 'R1', 'R2', 'R3', 'R4']

sns.heatmap(ExactAlign.PearsonMatrixFull[540:549, 540:549], cmap = 'RdBu_r', square = True,
            xticklabels = x_labels, yticklabels = y_labels, vmin = -1, vmax = 1) # 405:414

plt.xlabel('Reference', fontsize = 14)
plt.ylabel('Unknown', fontsize = 14)

sns.lineplot(x = x_values, y = y_values, color = 'black', linestyle = '-', linewidth = 1)
plt.suptitle("Anchor Point 2", fontsize = 18, y = 0.95, x = 0.455)
plt.savefig('heatmap_anchor2.png', dpi=300)
plt.show()
# 50 - 100: top left

## Anchor point 3

In [ ]:
# ExactAlign.mz_valueRef.iloc[ExactAlign.aligned_mz_clusters_ref[157]]
157*9

In [ ]:
PeakGroup.jointcluster[115]
115*9

In [ ]:
ExactAlign.mz_valueRef.iloc[ExactAlign.aligned_mz_clusters_ref[157]][0:8]

In [ ]:
ExactAlign.aligned_mz_clusters_ref[157]

In [ ]:
mz_wk2.iloc[ExactAlign.aligned_mz_clusters_ref[157]][0:8]

In [ ]:
mz_wk5.iloc[ExactAlign.aligned_mz_clusters_ref[153]][0:8]

In [ ]:
ExactAlign.mz_valueUnk.iloc[ExactAlign.aligned_mz_clusters_unk[157]][0:8]

In [ ]:
# ExactAlign.ref_clusters[157]
# ExactAlign.unk_clusters[157]

In [ ]:
PeakGroup.clusterUnk[143]

In [ ]:
PeakGroup.clusterRef[107]

In [ ]:
mz_wk2[5370:6600]

In [ ]:
ExactAlign.ref_clusters[157]

In [ ]:
ExactAlign.unk_clusters[157]

In [ ]:
y_labels = ['U-4', 'U-3', 'U-2', 'U-1', 'U0', 'U1', 'U2', 'U3', 'U4']
x_labels = ['R-4', 'R-3', 'R-2', 'R-1', 'R0', 'R1', 'R2', 'R3', 'R4']

sns.heatmap(ExactAlign.PearsonMatrixFull[1413:1422, 1413:1422], cmap = 'RdBu_r', square = True,
            xticklabels = x_labels, yticklabels = y_labels, vmin = -1, vmax = 1) # 1026:1035

plt.xlabel('Reference', fontsize = 14)
plt.ylabel('Unknown', fontsize = 14)

sns.lineplot(x = x_values, y = y_values, color = 'black', linestyle = '-', linewidth = 1)
plt.suptitle("Anchor Point 3", fontsize = 18, y = 0.95, x = 0.455)
plt.savefig('heatmap_anchor3.png', dpi=300)
plt.show()
# 10 - 60: left bottom

### Spectrum plot

In [ ]:
PeakGroup = gx.PeakCallingmv(MALDIdataAnn1, MALDIdataAnn2)
PeakGroup.peak_calling(0.9)
PeakGroup.peak_grouping(0.9)

In [ ]:
ExactAlign = gx.AnnDataMALDI(MALDIdataAnn1, MALDIdataAnn2)
ExactAlign.get_corr_peakgroup_refined(PeakGroup.jointcluster)
ExactAlign.peak_group_pairing()
ExactAlign.fine_alignment_assessment(threshold= 0.2, ignore = True)
ExactAlign.summarize()

In [ ]:
Aligned = gx.PGmzalign(ExactAlign)
MALDIdataAnn3 = Aligned.getAnnSim()

In [ ]:
sc.pp.normalize_per_cell(MALDIdataAnn3)

In [ ]:
meanspectrum_aligned = np.mean(MALDIdataAnn3.X, axis = 0)
meanspectrum_aligned_mv = np.convolve(meanspectrum_aligned, np.ones(3)/3, mode='valid')

In [ ]:
mz_aligned = MALDIdataAnn3.var

In [ ]:
mz_aligned_mv = mz_aligned[1:(mz_aligned.shape[0]-1)]
mz_aligned_mv 

In [ ]:
originaltable= pd.DataFrame(np.array(mz_aligned_mv["m/z"]))
originaltable = originaltable.rename(columns = {0:"mzaligned"})

In [ ]:
meanspectrum_aligned_mv

In [ ]:
originaltable= pd.DataFrame(np.array(mz_aligned_mv["m/z"]))
originaltable = originaltable.rename(columns = {0:"mzaligned"})
originaltable["intensityaligned"] = meanspectrum_aligned_mv
originaltable

In [ ]:
csv_file_path = f'{OUTPUT_DIR}_Maaike'
originaltable.to_csv (csv_file_path + "/mean_spec_mv_aligned.csv")